# Census Data Introduction

In this notebook we will:
1. Download Census data from the U.S. Census Bureau API
2. Clean and explore it
3. Join it with geographic boundaries (shapefiles)
4. Make maps and charts

**Before you start:** Look up the variable code(s) you want at  
https://api.census.gov/data/2023/acs/acs5/profile/variables.html  
Use **Ctrl+F** to search. Example: `DP04_0058E` = households without a vehicle.

Find your state FIPS code here: https://transition.fcc.gov/oet/info/maps/census/fips/fips.txt

Browse available shapefiles at: https://www2.census.gov/geo/tiger/TIGER2023/TRACT/

> **Stick with `tract` for this workshop** — later steps are built around tracts.

> As an optional part, we will convert tracts into counties and visualize that as well.

---
# Part 1: Download Census Data

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

## 1.1 — Set Your Query Options

Change the values below to match what you want to download.

In [ ]:
# CHANGE THESE
year       = "2023"         # ACS year (2009–2023)
variables  = "DP04_0058E"   # variable code(s), comma-separated
state_fips = "18"           # FIPS code for your state (18 = Indiana)
api_key    = ""             # optional — get one free at api.census.gov/data/key_signup.html

## 1.2 — Download the Data

This cell builds the Census API URL and loads the data directly into a DataFrame using pandas.

##### Note: If you have base url ends with `*&descriptive=true&outputFormat=csv`

Then you can:
```python
df = pd.read_csv(base_url)
df = df[df['NAME'] != 'Geographic Area Name'] # remove the first row
```

In [ ]:
base_url = f"https://api.census.gov/data/{year}/acs/acs5/profile"
url = f"{base_url}?get=NAME,GEO_ID,{variables}&for=tract:*&in=state:{state_fips}%20county:*"

if api_key:
    url += f"&key={api_key}"

df = pd.read_json(url)
df.columns = df.iloc[0]       # first row is the header
df = df[1:].reset_index(drop=True)

print(f"Downloaded {df.shape[0]} rows and {df.shape[1]} columns")
df.head()

## 1.3 — Clean the GEOID Column

The raw `GEO_ID` looks like `1400000US18001`. For maps we only need the part after `US` (e.g. `18001950100`).

In [ ]:
df["GEOID"] = df["GEO_ID"].str.split("US").str[1]

print("Sample GEOID values:", df["GEOID"].head().tolist())
df.head()

---
# Part 2: Explore and Clean the Data

## 2.1 — Inspect the Columns

`.info()` shows column names, data types, and how many values are non-null.

In [ ]:
df.info()

## 2.2 — Convert to Numbers and Remove Missing Values

The Census API sends everything as text. We convert estimate columns to real numbers, then drop any rows with missing values.

In [ ]:
# Find all estimate columns (they end with 'E', excluding name/ID columns)
estimate_cols = [c for c in df.columns if c.endswith("E") and c not in ("NAME", "GEO_ID", "GEOID")]

# Convert to numbers (invalid values become NaN)
df[estimate_cols] = df[estimate_cols].apply(pd.to_numeric, errors="coerce")

# Drop rows with missing values
df_clean = df.dropna(subset=estimate_cols).copy()

print(f"{len(df)} rows before, {len(df_clean)} rows after removing missing values")
df_clean[estimate_cols].describe().round(1)

## 2.3 — Quick Stats for Your Variable

Update `variable` below if you downloaded a different code.

In [ ]:
variable  = "DP04_0058E"                   # ← update to your variable
var_label = "Households Without a Vehicle" # ← friendly name for charts

col = df_clean[variable]
print(f"Variable : {variable}")
print(f"Min      : {col.min():.0f}")
print(f"Max      : {col.max():.0f}")
print(f"Mean     : {col.mean():.1f}")
print(f"Median   : {col.median():.1f}")
print(f"Missing  : {col.isna().sum()} rows")

---
# Part 3: Load the Shapefile and Join

Census data tells us *numbers* per area. Shapefiles tell us the *boundaries* of those areas.  
We download/read the shapefile and merge the two together.


## 3.1 — Load the Shapefile

GeoPandas can read a `.zip` file directly from a URL — no manual downloading or extracting needed.

In [ ]:
fips_code = 18    # ← update to your state FIPS

zip_url = f"https://www2.census.gov/geo/tiger/TIGER{year}/TRACT/tl_{year}_{fips_code:02d}_tract.zip"

gdf = gpd.read_file(zip_url)
print(f"Loaded {len(gdf)} tracts")
gdf.head()

## 3.2 — Join Shapefile with Census Data

We match rows using the `GEOID` column — it is the shared key between the two datasets.

In [ ]:
merged_gdf = gdf.merge(df_clean, on="GEOID", how="inner")

print(f"After join: {len(merged_gdf)} tracts matched")
merged_gdf.head()

## 3.3 — Top Counties

The first 5 digits of a tract GEOID identify the county. We group by those digits to get county totals.

In [ ]:
merged_gdf["county_fips"] = merged_gdf["GEOID"].str[:5]

county_totals = (
    merged_gdf.groupby("county_fips")[variable]
    .sum()
    .reset_index()
    .rename(columns={variable: "total"})
    .sort_values("total", ascending=False)
)

print("Top 10 counties:")
county_totals.head(10)

---
# Part 4: Visualize the Data

## 4.1 — Choropleth Map

Each census tract is colored by its value. Darker = higher.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

merged_gdf.plot(
    column=variable,
    cmap="YlOrRd",
    linewidth=0.15,
    edgecolor="grey",
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "No data"},
    ax=ax
)

ax.set_title(f"{var_label} by Census Tract ({year} ACS)", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

## 4.2 — Bar Chart: Top 15 Counties

In [ ]:
top15 = county_totals.head(15).sort_values("total")

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top15["county_fips"], top15["total"], color="steelblue")

ax.set_xlabel(var_label)
ax.set_title(f"Top 15 Counties — {var_label}")
ax.bar_label(bars, fmt="{:,.0f}", padding=4)

plt.tight_layout()
plt.show()

## 4.3 — Histogram

This shows how the values are spread across all tracts. The red line is the median, orange is the mean.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(merged_gdf[variable].dropna(), bins=40, color="steelblue", edgecolor="white")

median_val = merged_gdf[variable].median()
mean_val   = merged_gdf[variable].mean()

ax.axvline(median_val, color="red",    linestyle="--", label=f"Median: {median_val:.0f}")
ax.axvline(mean_val,   color="orange", linestyle="--", label=f"Mean:   {mean_val:.0f}")

ax.set_xlabel(var_label)
ax.set_ylabel("Number of Census Tracts")
ax.set_title(f"Distribution of {var_label} by Tract")
ax.legend()

plt.tight_layout()
plt.show()

## 4.4 — Classifying Tracts: Above vs. Below Average

A common task is to split data into groups based on a threshold.  
Here we label each tract as **Above** or **Below** the state average.

In [ ]:
avg = merged_gdf[variable].mean()
print(f"Average tract value for {variable}: {avg:.1f}")

merged_gdf["Class"] = ["Above" if v > avg else "Below" for v in merged_gdf[variable]]

print(merged_gdf["Class"].value_counts())

In [ ]:
merged_gdf["color"] = merged_gdf["Class"].map({"Above": "green", "Below": "red"})
# plot the above v below map

---
**All done!** You have downloaded, cleaned, joined, and visualized Census data.  
Try swapping in a different variable code at the top of Part 1 and run through again to compare.


# Optional — Part 5: Dissolve Tracts into Counties

So far our data is at the **tract** level — thousands of small polygons.  
Sometimes it's more useful to work at the **county** level instead.

**Dissolving** merges all tract polygons that share the same county FIPS code into a single county polygon, and aggregates the data values (here: summing them up).

## 5.1 — Dissolve Tracts into County Polygons

We group by `county_fips` and sum the variable. GeoPandas automatically merges the geometries too.

In [ ]:
# Make sure county_fips column exists (created in Part 3.3)
merged_gdf["county_fips"] = merged_gdf["GEOID"].str[:5]

county_gdf = merged_gdf[["county_fips", variable, "geometry"]].dissolve(   # we will use the dissolve function
    by="county_fips",
    aggfunc="sum"
).reset_index()

print(f"Dissolved {len(merged_gdf)} tracts into {len(county_gdf)} counties")
county_gdf.head()

## 5.2 — Choropleth Map at the County Level

Same map as before, but now each county is one polygon colored by its total value.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

county_gdf.plot(
    column=variable,
    cmap="YlOrRd",
    linewidth=0.5,
    edgecolor="grey",
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "No data"},
    ax=ax
)

ax.set_title(f"{var_label} by County ({year} ACS)", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

## 5.3 — Side-by-Side: Tract vs. County

Plotting both maps together makes it easy to see how dissolving changes the level of detail.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Tract-level map
merged_gdf.plot(
    column=variable, cmap="YlOrRd", linewidth=0.1,
    edgecolor="grey", legend=True,
    missing_kwds={"color": "lightgrey", "label": "No data"},
    ax=ax1
)
ax1.set_title(f"{var_label}\nby Census Tract", fontsize=13)
ax1.axis("off")

# County-level map
county_gdf.plot(
    column=variable, cmap="YlOrRd", linewidth=0.5,
    edgecolor="grey", legend=True,
    missing_kwds={"color": "lightgrey", "label": "No data"},
    ax=ax2
)
ax2.set_title(f"{var_label}\nby County (Dissolved)", fontsize=13)
ax2.axis("off")

plt.suptitle(f"{year} ACS — Tract vs. County Level", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()